In [ ]:
%%sql -r dataframe_1
USE ROLE PROJECT_ROLE;
USE WAREHOUSE COMPUTE_WH;

SELECT
    f.traffic_density,
    f.horn_events_per_min,
    f.avg_speed,
    w.weather_condition,
    r.road_quality_score,
    d.driver_experience_level,
    f.stress_index
FROM TRAFFIC_PROJECT.ANALYTICS.FACT_TRAFFIC f
JOIN TRAFFIC_PROJECT.ANALYTICS.DIM_WEATHER w
    ON f.weather_id = w.weather_id
JOIN TRAFFIC_PROJECT.ANALYTICS.DIM_DRIVER d
    ON f.driver_id = d.driver_id
JOIN TRAFFIC_PROJECT.ANALYTICS.DIM_ROAD_QUALITY r
    ON f.road_quality_id = r.road_quality_id
LIMIT 10;

In [ ]:
%%sql -r dataframe_2
LIST @ML_ARTIFACTS_DB.PREPROCESSING.MODEL_ARTIFACTS
PATTERN = '.*\\.pkl';

Do budowy wszystkich trzech modeli wykorzystano ten sam preprocessor, aby zachować spójność procesu modelowania i uczciwie porównać wyniki modeli. Preprocessor powinien być zgodny z finalnym etapem przygotowania danych wykonanym przez Agnieszkę. Oznacza to, że w modelowaniu pozostaje zmienna `traffic_density`, natomiast zmienna `signal_wait_time` zostaje pominięta z powodu silnej współliniowości. W modelach nie jest używana zmienna `congestion_index`.

In [ ]:
from snowflake.snowpark.context import get_active_session

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Połączenie z aktywną sesją Snowflake

session = get_active_session()


# Pobranie danych ze Snowflake

query = """
SELECT
    f.traffic_density,
    f.horn_events_per_min,
    f.avg_speed,
    w.weather_condition,
    r.road_quality_score,
    d.driver_experience_level,
    f.stress_index
FROM TRAFFIC_PROJECT.ANALYTICS.FACT_TRAFFIC f
JOIN TRAFFIC_PROJECT.ANALYTICS.DIM_WEATHER w
    ON f.weather_id = w.weather_id
JOIN TRAFFIC_PROJECT.ANALYTICS.DIM_DRIVER d
    ON f.driver_id = d.driver_id
JOIN TRAFFIC_PROJECT.ANALYTICS.DIM_ROAD_QUALITY r
    ON f.road_quality_id = r.road_quality_id
"""

df = session.sql(query).to_pandas()

# Snowflake często zwraca nazwy kolumn wielkimi literami,
# dlatego zmieniamy je na małe litery.

df.columns = [col.lower() for col in df.columns]

# Ujednolicenie wartości tekstowych.

df["weather_condition"] = df["weather_condition"].str.strip().str.title()
df["driver_experience_level"] = df["driver_experience_level"].str.strip().str.title()

print("Podgląd danych:")
print(df.head())

print("\nKolumny w danych:")
print(df.columns.tolist())

print("\nLiczba rekordów:", len(df))


# Wybór zmiennych do modelu

feature_columns = [
    "horn_events_per_min",
    "avg_speed",
    "road_quality_score",
    "traffic_density",
    "weather_condition",
    "driver_experience_level"
]

target_column = "stress_index"

X = df[feature_columns]
y = df[target_column]

print("\nPodgląd zmiennych X:")
print(X.head())

print("\nPodgląd zmiennej y:")
print(y.head())


# Podział danych na zbiór treningowy i testowy

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nRozmiary zbiorów:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


# Pobranie preprocessora ze Snowflake Stage

stage_name = "@ML_ARTIFACTS_DB.PREPROCESSING.MODEL_ARTIFACTS"

preprocessor_filename = "preprocessor_20260514_202212.pkl"
preprocessor_pattern = ".*preprocessor_20260514_202212[.]pkl"

local_dir = "/tmp/model_artifacts"
os.makedirs(local_dir, exist_ok=True)

print("\nSzukam preprocessora na Stage...")

files_on_stage = session.sql(f"""
    LIST {stage_name}
    PATTERN = '{preprocessor_pattern}'
""").collect()

if len(files_on_stage) == 0:
    raise FileNotFoundError("Nie znaleziono preprocessora na Stage.")

print("Znaleziono plik na Stage:")

for row in files_on_stage:
    print(row["name"])

print("\nPobieram preprocessor ze Stage...")

session.file.get(
    stage_name,
    local_dir,
    pattern=preprocessor_pattern
)

local_path = None

for root, dirs, files in os.walk(local_dir):
    for file in files:
        if file == preprocessor_filename:
            local_path = os.path.join(root, file)

if local_path is None:
    raise FileNotFoundError("Nie znaleziono pliku preprocessora po pobraniu.")

preprocessor = joblib.load(local_path)

print("\nPreprocessor został wczytany.")
print(preprocessor)


# Prosta kontrola preprocessora

all_preprocessor_columns = []

for name, transformer, columns in preprocessor.transformers_:
    if isinstance(columns, str):
        columns = [columns]
    else:
        columns = list(columns)

    all_preprocessor_columns.extend(columns)

print("\nKolumny używane przez preprocessor:")
print(all_preprocessor_columns)

if "traffic_density" not in all_preprocessor_columns:
    raise ValueError("Preprocessor nie zawiera kolumny traffic_density.")

if "signal_wait_time" in all_preprocessor_columns:
    raise ValueError("Preprocessor nadal zawiera kolumnę signal_wait_time.")

print("\nPreprocessor jest zgodny z założeniami rozdziału 3.")
print("Zawiera traffic_density i nie zawiera signal_wait_time.")


# Transformacja danych

X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print("\nDane po preprocessingu:")
print("X_train_proc:", X_train_proc.shape)
print("X_test_proc:", X_test_proc.shape)


# Utworzenie modeli predykcyjnych

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


# Trenowanie modeli i obliczenie metryk

results = []

for model_name, model in models.items():

    print("\nTrenuję model:", model_name)

    model.fit(X_train_proc, y_train)

    y_train_pred = model.predict(X_train_proc)
    y_test_pred = model.predict(X_test_proc)

    mae_train = mean_absolute_error(y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    r2_train = r2_score(y_train, y_train_pred)

    mae_test = mean_absolute_error(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2_test = r2_score(y_test, y_test_pred)

    results.append({
        "model": model_name,
        "MAE_train": mae_train,
        "MAE_test": mae_test,
        "RMSE_train": rmse_train,
        "RMSE_test": rmse_test,
        "R2_train": r2_train,
        "R2_test": r2_test
    })

    print("MAE test:", mae_test)
    print("RMSE test:", rmse_test)
    print("R2 test:", r2_test)


# Porównanie modeli

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="RMSE_test",
    ascending=True
).reset_index(drop=True)

results_df_rounded = results_df.round(4)

print("\nTabela porównawcza modeli:")
print(results_df_rounded)

best_model_name = results_df.loc[0, "model"]
best_model = models[best_model_name]

print("\nNajlepszy model według RMSE_test:")
print(best_model_name)

results_df_rounded

In [ ]:
# Zapis najlepszego modelu na Snowflake Stage

import os
import joblib

local_dir = "/tmp/model_artifacts"
os.makedirs(local_dir, exist_ok=True)

# Wybieramy najlepszy model automatycznie na podstawie RMSE_test.
best_model_name = results_df.loc[results_df["RMSE_test"].idxmin(), "model"]
best_model = models[best_model_name]

print("Najlepszy model do zapisu:", best_model_name)

model_filename = "traffic_stress_best_model.pkl"
local_model_path = os.path.join(local_dir, model_filename)

joblib.dump(best_model, local_model_path)

print("Model zapisany lokalnie:")
print(local_model_path)

stage_path = "@ML_ARTIFACTS_DB.PREPROCESSING.MODEL_ARTIFACTS"

session.file.put(
    local_model_path,
    stage_path,
    overwrite=True,
    auto_compress=False
)

print("Model został zapisany na Snowflake Stage:")
print(stage_path + "/" + model_filename)